# Normalization Lesson (6/19/26)

---

*Codecademy — MLE Path: Supervised Learning I (Regressors, Classifiers, Trees). Concise grad-student notes.*

## Why normalize?

- **Core problem:** features live on wildly different *scales*. e.g. a `salary` feature (10,000–100,000) vs an `age` feature (18–70).
- Many algorithms treat features as living in a shared geometric space. If one feature has a huge numeric range, it **dominates** the others — not because it's more important, but because its units are bigger.
- **Who cares about scale (normalize these):**
  - **Distance-based** models: KNN, K-Means, SVM (RBF) — they compute Euclidean distance, so large-range features swamp the distance metric.
  - **Gradient descent** learners: linear/logistic regression, neural nets — unscaled features make the loss surface elongated → slow / unstable convergence.
  - **Regularized** models (Ridge/Lasso): the penalty is applied per-coefficient, so it's only fair if features share a scale.
- **Who doesn't care (scale-invariant):** tree-based models (Decision Trees, Random Forest, Gradient Boosting). They split on thresholds *within* a single feature, so monotonic rescaling changes nothing.

> **Rule of thumb:** normalization rescales features to a comparable range so no single feature dominates due to units alone.

## The two workhorses

| Method | Formula | Output range | Outlier sensitivity |
|---|---|---|---|
| **Min-Max Normalization** | $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$ | $[0, 1]$ (bounded) | **High** — min/max are set by extremes |
| **Z-Score Standardization** | $x' = \dfrac{x - \mu}{\sigma}$ | unbounded, mean 0, std 1 | **Lower** — uses mean & std, not extremes |

("Standardization" and "normalization" get used loosely; here Min-Max = normalization, Z-score = standardization.)

## 1) Min-Max Normalization

Linearly rescales each feature into **[0, 1]**. The smallest value maps to 0, the largest to 1, everything else proportionally between.

$$x'_i = \frac{x_i - \min(x)}{\max(x) - \min(x)}$$

- **Pro:** guaranteed bounded range — nice when an algorithm expects inputs in [0,1] (e.g. some NN activations, image pixels).
- **Con:** a single extreme outlier sets `max` (or `min`), squashing all the "normal" points into a tiny sub-interval.

In [1]:
import numpy as np

ages = np.array([18, 25, 32, 47, 70], dtype=float)

def min_max(x):
    return (x - x.min()) / (x.max() - x.min())

print("raw:       ", ages)
print("min-max:   ", np.round(min_max(ages), 3))
# -> min value (18) becomes 0.0, max value (70) becomes 1.0


raw:        [18. 25. 32. 47. 70.]
min-max:    [0.    0.135 0.269 0.558 1.   ]


**Outlier demo** — watch one extreme value crush everything else:

In [2]:
ages_outlier = np.array([18, 25, 32, 47, 70, 500], dtype=float)  # 500 = data-entry error
print("min-max w/ outlier:", np.round(min_max(ages_outlier), 3))
# The first 5 real ages now all sit near 0 — info between them is lost.


min-max w/ outlier: [0.    0.015 0.029 0.06  0.108 1.   ]


## 2) Z-Score Standardization

Recenters each feature to **mean 0** and rescales to **std 1**. Result = how many standard deviations a point sits from the mean.

$$x'_i = \frac{x_i - \mu}{\sigma}$$

- **Pro:** far less sensitive to outliers (mean/std move less than min/max); preserves the *shape* of the distribution.
- **Con:** output is **not bounded** — values can be negative or > 1, which some algorithms dislike.
- This is the usual default for regression / gradient-descent models.

In [3]:
def z_score(x):
    return (x - x.mean()) / x.std()

print("raw:       ", ages)
print("z-score:   ", np.round(z_score(ages), 3))
print("mean ~0:   ", round(z_score(ages).mean(), 6))
print("std  ~1:   ", round(z_score(ages).std(),  6))


raw:        [18. 25. 32. 47. 70.]
z-score:    [-1.103 -0.725 -0.346  0.465  1.709]
mean ~0:    0.0
std  ~1:    1.0


## Doing it the real way: scikit-learn

In practice you use `sklearn.preprocessing`. Key API: **`.fit()` learns the scaling params from TRAIN, `.transform()` applies them.**

- `MinMaxScaler` → min-max normalization
- `StandardScaler` → z-score standardization

In [4]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

X = ages.reshape(-1, 1)  # sklearn expects 2D: (n_samples, n_features)

print("MinMaxScaler:  ", np.round(MinMaxScaler().fit_transform(X).ravel(), 3))
print("StandardScaler:", np.round(StandardScaler().fit_transform(X).ravel(), 3))


MinMaxScaler:   [0.    0.135 0.269 0.558 1.   ]
StandardScaler: [-1.103 -0.725 -0.346  0.465  1.709]


## ⚠️ The cardinal rule: fit on TRAIN only

Fit the scaler on the **training set**, then `transform` both train and test with those *same* parameters.

- Calling `fit` (or `fit_transform`) on the test set, or on the full dataset before splitting, leaks test-set statistics (its min/max/mean/std) into training → **data leakage** → optimistic, dishonest scores.
- At inference time you won't *have* the test min/max anyway, so train-derived params are the only honest choice.

In [5]:
from sklearn.model_selection import train_test_split

X_full = ages.reshape(-1, 1)
X_train, X_test = train_test_split(X_full, test_size=0.4, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # FIT on train
X_test_s  = scaler.transform(X_test)         # TRANSFORM test with train params (no re-fit!)

print("learned mean/std from train:", scaler.mean_.round(2), scaler.scale_.round(2))
print("train scaled:", X_train_s.ravel().round(3))
print("test  scaled:", X_test_s.ravel().round(3))


learned mean/std from train: [32.33] [11.84]
train scaled: [-0.028 -1.21   1.239]
test  scaled: [-0.619  3.181]


## TL;DR

- **Normalize when** the model is distance-based, gradient-descent-based, or regularized. **Skip** for tree-based models.
- **Min-Max** → bounded [0,1], simple, but outlier-fragile.
- **Z-Score** → mean 0 / std 1, unbounded, outlier-robust, the safe default.
- **Always** `fit` the scaler on train data only; `transform` everything with those params to avoid leakage.